# Imbalance-Ratio Results Table

Generate paper-ready class-imbalance-ratio (IR) comparisons from completed `with_c` benchmark batches.

The notebook validates all ten batches and 30 datasets. It never recomputes models or modifies CSV results. It writes one table comparing the mean training-fold IR before resampling with IR after Re-SC and KMeans-ReSC.

## Load and validate benchmark results

Run this cell first. It loads all completed `with_c` outer-fold files, removes duplicated classifier rows, and verifies 100 distinct training folds per dataset and reported method.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

WORKING_DIRECTORY = Path.cwd().resolve()
root_candidates = (
    WORKING_DIRECTORY,
    WORKING_DIRECTORY.parent,
    WORKING_DIRECTORY / 'resc-experiments',
)
EXPERIMENT_ROOT = next(
    (candidate for candidate in root_candidates if (candidate / 'results' / 'metrics').exists()),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError('Run from resc-experiments or its notebooks directory.')

METRICS_ROOT = EXPERIMENT_ROOT / 'results' / 'metrics'
LATEX_OUTPUT = EXPERIMENT_ROOT.parent / 'latex' / 'tables' / 'additional_diagnostics_results.tex'
EXPECTED_BATCHES = 10
EXPECTED_DATASETS = 30
EXPECTED_FOLDS = 100
REPORTED_METHODS = ('none', 'resc', 'kmeans_resc_q10')


def batch_number(path: Path) -> int:
    match = re.search(r'batch(\d+)of10', str(path))
    if match is None:
        raise ValueError(f'Cannot identify batch number from {path}')
    return int(match.group(1))


def load_batched(pattern: str) -> pd.DataFrame:
    paths = sorted(METRICS_ROOT.glob(pattern), key=batch_number)
    if len(paths) != EXPECTED_BATCHES:
        raise FileNotFoundError(f'{pattern}: expected {EXPECTED_BATCHES} files, found {len(paths)}')
    if [batch_number(path) for path in paths] != list(range(1, EXPECTED_BATCHES + 1)):
        raise ValueError(f'{pattern}: missing or duplicate batch numbers')
    frame = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    if frame['dataset_key'].nunique() != EXPECTED_DATASETS:
        raise ValueError(f'{pattern}: expected {EXPECTED_DATASETS} datasets, found {frame["dataset_key"].nunique()}')
    return frame


outer_folds = load_batched(
    'benchmark_fixed_m15_with_c_10x10_batch*/with_c/outer_fold_metrics.csv'
)

# Each resampling result is repeated once per classifier. Counts do not depend on
# classifier, so retain one row per dataset, repeat, fold, and method.
fold_counts = outer_folds.loc[
    outer_folds['method'].isin(REPORTED_METHODS)
].drop_duplicates(subset=['dataset_key', 'repeat', 'fold', 'method']).copy()

expected = pd.MultiIndex.from_product(
    [fold_counts['dataset_key'].unique(), REPORTED_METHODS],
    names=['dataset_key', 'method'],
)
observed = fold_counts.groupby(['dataset_key', 'method']).size().reindex(expected)
if observed.isna().any() or not observed.eq(EXPECTED_FOLDS).all():
    bad = observed.loc[observed.isna() | ~observed.eq(EXPECTED_FOLDS)].to_dict()
    raise ValueError(f'Expected {EXPECTED_FOLDS} folds per dataset and method: {bad}')

print(f'Loaded {len(fold_counts):,} distinct resampling records for {EXPECTED_DATASETS} datasets.')
print(f'LaTeX output: {LATEX_OUTPUT}')


## Compute and format IR comparisons

For each training fold, IR is `resampled_majority / resampled_minority`. For `Original`, resampled counts equal the unresampled training counts. Each table entry is the mean of 100 fold-level ratios for one dataset.

In [ ]:
METHOD_LABELS = {
    'none': 'Before resampling',
    'resc': 'After Re-SC',
    'kmeans_resc_q10': 'After KMeans-ReSC',
}


def latex_escape(value: object) -> str:
    return str(value).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('&', r'\&').replace('%', r'\%')


def dataset_name(value: str) -> str:
    clean = re.sub(r'^(?:imblearn_|openml_task_\d+_|uci_|keel_)', '', value)
    return latex_escape(clean)


def fmt_ir(value: object) -> str:
    if pd.isna(value):
        return '--'
    return f'{float(value):.2f}'


def latex_table(headers, rows, caption, label, column_format):
    lines = [
        r'\begin{table}[!t]',
        r'\centering',
        rf'\caption{{{caption}}}',
        rf'\label{{{label}}}',
        r'\setlength{\tabcolsep}{1.6pt}',
        r'\renewcommand{\arraystretch}{0.78}',
        r'\scriptsize',
        rf'\begin{{tabular}}{{{column_format}}}',
        r'\toprule',
        ' & '.join(headers) + r' \\',
        r'\midrule',
    ]
    lines.extend(' & '.join(row) + r' \\' for row in rows)
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    return '\n'.join(lines)


fold_counts['ir'] = (
    fold_counts['resampled_majority'] / fold_counts['resampled_minority']
)
ir_by_dataset = (
    fold_counts.groupby(['dataset_key', 'method'], as_index=False)['ir']
    .mean()
    .pivot(index='dataset_key', columns='method', values='ir')
    .reindex(columns=REPORTED_METHODS)
)

if ir_by_dataset.isna().any().any():
    raise ValueError('Missing an IR value for at least one dataset-method combination.')

# Present datasets in ascending pre-resampling IR, as in a conventional benchmark
# characteristics table.
ir_by_dataset = ir_by_dataset.sort_values('none', kind='stable')
ir_rows = [
    [
        str(number),
        dataset_name(dataset_key),
        *(fmt_ir(values[method]) for method in REPORTED_METHODS),
    ]
    for number, (dataset_key, values) in enumerate(ir_by_dataset.iterrows(), start=1)
]
ir_table = latex_table(
    ['No.', 'Dataset', 'Before', 'Re-SC', 'KMeans-ReSC'],
    ir_rows,
    r'Class imbalance ratio ($IR=n_{\mathcal{N}}/n_{\mathcal{P}}$) before and after resampling. Each entry is the mean of 100 training-fold ratios. Values closer to 1 indicate better class balance; $IR<1$ indicates reversal of the original class-size ordering.',
    'tab:imbalance_ratio_before_after',
    '@{}rlrrr@{}',
)

latex_output = ir_table + '\n'
display(Markdown(f'Generated **one** IR comparison table from **{len(ir_by_dataset)}** datasets.'))


## Write and display LaTeX

Run this cell after reviewing the table. It writes only the IR comparison table. Include it with `\input{tables/additional_diagnostics_results}` in `latex/main.tex` when ready.

In [ ]:
LATEX_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
LATEX_OUTPUT.write_text(latex_output, encoding='utf-8')
print(latex_output)
print(f'\nSaved: {LATEX_OUTPUT}')